# Evaluation 05 — Box plots por grupo de papel e agregado

Notebook para comparar vários modelos usando box plots por grupo de papel e também no agregado.

Edite apenas o vetor `registra_modelo` e, se necessário, os nomes das colunas em `CONFIG`.

In [ ]:
# ============================================================
# 1. Imports e configuração geral
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Para evitar erro de permissão em clusters/servidores
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ------------------------------------------------------------
# CONFIGURAÇÃO PRINCIPAL
# ------------------------------------------------------------

CONFIG = {
    # Colunas esperadas nos arquivos de avaliação/previsão.
    # O notebook tenta inferir alternativas quando estas não existirem.
    "col_papel": "papel",
    "col_grupo": "grupo_papel",
    "col_data": "data",
    "col_horizonte": "horizonte",
    "col_y_true": "y_true",
    "col_y_pred": "y_pred",
    
    # Métrica padrão para os box plots.
    # Opções calculáveis: "erro", "abs_error", "squared_error", "ape"
    # Ou uma coluna já existente, por exemplo "mae", "mse", "rmse".
    "metrica": "abs_error",
    
    # Padrões de arquivos procurados dentro de cada endereço.
    "padroes_arquivos": [
        "*.parquet", "*.csv", "*.feather", "*.pkl", "*.pickle"
    ],
    
    # Nomes de arquivos preferidos quando o endereço for uma pasta.
    "arquivos_preferidos": [
        "avaliacao.parquet", "avaliacao.csv",
        "evaluation.parquet", "evaluation.csv",
        "previsoes.parquet", "previsoes.csv",
        "predictions.parquet", "predictions.csv",
        "forecast.parquet", "forecast.csv",
    ],
    
    # Colunas mínimas para manter, quando existirem.
    "colunas_extra": []
}

In [ ]:
# ============================================================
# 2. Registro dos modelos
# ============================================================
# Edite este vetor. Cada modelo deve apontar para um arquivo ou pasta.
#
# Exemplo:
# registra_modelo = [
#     {
#         "nome": "DLinear",
#         "endereco": "../previsoes/b3_daily_financeiro/DLinear/2026-06-01",
#         "cor": "#1f77b4",
#     },
#     {
#         "nome": "TransformerCI",
#         "endereco": "../previsoes/b3_daily_financeiro/TransformerChannelIndependent/2026-06-01",
#         "cor": "#ff7f0e",
#     },
# ]

registra_modelo = [
    # {"nome": "Modelo_1", "endereco": "CAMINHO/DO/MODELO_1", "cor": "#1f77b4"},
    # {"nome": "Modelo_2", "endereco": "CAMINHO/DO/MODELO_2", "cor": "#ff7f0e"},
]

# Caso a cor seja omitida, o notebook atribui automaticamente.
PALETA_PADRAO = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
# ============================================================
# 3. Funções auxiliares de leitura
# ============================================================

def _ler_arquivo(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".feather":
        return pd.read_feather(path)
    if suffix in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    raise ValueError(f"Extensão não suportada: {path}")


def _arquivos_candidatos(endereco: str | Path, config: dict) -> list[Path]:
    p = Path(endereco).expanduser()
    
    if p.is_file():
        return [p]
    
    if not p.exists():
        raise FileNotFoundError(f"Endereço não encontrado: {p}")
    
    preferidos = []
    for nome in config["arquivos_preferidos"]:
        preferidos.extend(list(p.rglob(nome)))
    if preferidos:
        return sorted(set(preferidos))
    
    candidatos = []
    for padrao in config["padroes_arquivos"]:
        candidatos.extend(list(p.rglob(padrao)))
    return sorted(set(candidatos))


def carregar_modelo(registro: dict, config: dict) -> pd.DataFrame:
    nome = registro["nome"]
    endereco = registro["endereco"]
    
    arquivos = _arquivos_candidatos(endereco, config)
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo de resultado encontrado para {nome}: {endereco}")
    
    partes = []
    for arq in arquivos:
        try:
            df = _ler_arquivo(arq)
            df["_arquivo_origem"] = str(arq)
            partes.append(df)
        except Exception as e:
            print(f"[AVISO] Ignorando {arq}: {e}")
    
    if not partes:
        raise RuntimeError(f"Nenhum arquivo legível para {nome}: {endereco}")
    
    out = pd.concat(partes, ignore_index=True)
    out["modelo"] = nome
    return out


def carregar_todos_modelos(registra_modelo: list[dict], config: dict) -> pd.DataFrame:
    if not registra_modelo:
        raise ValueError("Preencha o vetor registra_modelo antes de executar.")
    
    frames = []
    for i, reg in enumerate(registra_modelo):
        reg.setdefault("cor", PALETA_PADRAO[i % len(PALETA_PADRAO)])
        frames.append(carregar_modelo(reg, config))
    
    return pd.concat(frames, ignore_index=True)

In [ ]:
# ============================================================
# 4. Padronização de colunas e cálculo de métricas
# ============================================================

def escolher_coluna(df: pd.DataFrame, preferida: str, alternativas: list[str]) -> str | None:
    if preferida in df.columns:
        return preferida
    
    lower_map = {c.lower(): c for c in df.columns}
    for alt in alternativas:
        if alt in df.columns:
            return alt
        if alt.lower() in lower_map:
            return lower_map[alt.lower()]
    return None


def padronizar_e_metricas(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()
    
    col_papel = escolher_coluna(df, config["col_papel"], ["ticker", "ativo", "symbol", "codigo", "codneg", "papel"])
    col_grupo = escolher_coluna(df, config["col_grupo"], ["grupo", "grupo_papel", "classe", "setor", "tipo_papel"])
    col_data = escolher_coluna(df, config["col_data"], ["date", "datetime", "timestamp", "data"])
    col_h = escolher_coluna(df, config["col_horizonte"], ["h", "horizon", "step", "passo", "horizonte"])
    col_y = escolher_coluna(df, config["col_y_true"], ["y", "target", "real", "observado", "valor_real", "y_true"])
    col_pred = escolher_coluna(df, config["col_y_pred"], ["pred", "prediction", "forecast", "previsto", "valor_previsto", "y_pred"])
    
    ren = {}
    if col_papel: ren[col_papel] = "papel"
    if col_grupo: ren[col_grupo] = "grupo_papel"
    if col_data: ren[col_data] = "data"
    if col_h: ren[col_h] = "horizonte"
    if col_y: ren[col_y] = "y_true"
    if col_pred: ren[col_pred] = "y_pred"
    df = df.rename(columns=ren)
    
    # Grupo padrão caso não exista coluna de grupo.
    if "grupo_papel" not in df.columns:
        if "papel" in df.columns:
            # Heurística B3: sufixo numérico do ticker.
            df["grupo_papel"] = df["papel"].astype(str).str.extract(r"(\d+)$", expand=False).fillna("sem_grupo")
            df["grupo_papel"] = "tipo_" + df["grupo_papel"].astype(str)
        else:
            df["grupo_papel"] = "sem_grupo"
    
    if "papel" not in df.columns:
        df["papel"] = "sem_papel"
    
    # Métricas por observação, quando y_true/y_pred existirem.
    if {"y_true", "y_pred"}.issubset(df.columns):
        df["erro"] = df["y_pred"] - df["y_true"]
        df["abs_error"] = df["erro"].abs()
        df["squared_error"] = df["erro"] ** 2
        den = df["y_true"].abs().replace(0, np.nan)
        df["ape"] = df["abs_error"] / den
    
    metrica = config["metrica"]
    if metrica not in df.columns:
        raise ValueError(
            f"Métrica '{metrica}' não existe. Colunas disponíveis: {sorted(df.columns.tolist())}"
        )
    
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[metrica, "modelo", "grupo_papel"])
    return df


df_raw = carregar_todos_modelos(registra_modelo, CONFIG)
df_eval = padronizar_e_metricas(df_raw, CONFIG)

print(df_eval.shape)
display(df_eval.head())

In [ ]:
# ============================================================
# 5. Resumo tabular
# ============================================================

metrica = CONFIG["metrica"]

resumo_grupo = (
    df_eval
    .groupby(["grupo_papel", "modelo"], observed=True)[metrica]
    .agg(n="count", media="mean", mediana="median", desvio="std", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75))
    .reset_index()
    .sort_values(["grupo_papel", "mediana"])
)

resumo_agregado = (
    df_eval
    .groupby(["modelo"], observed=True)[metrica]
    .agg(n="count", media="mean", mediana="median", desvio="std", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75))
    .reset_index()
    .sort_values("mediana")
)

display(resumo_agregado)
display(resumo_grupo)

In [ ]:
# ============================================================
# 6. Box plot agregado
# ============================================================

def mapa_cores(registra_modelo):
    return {reg["nome"]: reg.get("cor", PALETA_PADRAO[i % len(PALETA_PADRAO)]) for i, reg in enumerate(registra_modelo)}

CORES = mapa_cores(registra_modelo)

def boxplot_modelos(ax, df, titulo, metrica, ordem_modelos=None, cores=None):
    if ordem_modelos is None:
        ordem_modelos = df.groupby("modelo")[metrica].median().sort_values().index.tolist()
    
    dados = [df.loc[df["modelo"] == m, metrica].dropna().values for m in ordem_modelos]
    bp = ax.boxplot(
        dados,
        labels=ordem_modelos,
        patch_artist=True,
        showfliers=False,
        medianprops={"linewidth": 2},
    )
    
    for patch, modelo in zip(bp["boxes"], ordem_modelos):
        if cores and modelo in cores:
            patch.set_facecolor(cores[modelo])
        patch.set_alpha(0.65)
    
    ax.set_title(titulo)
    ax.set_ylabel(metrica)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)
    return ax

fig, ax = plt.subplots(figsize=(10, 5))
ordem_modelos = resumo_agregado["modelo"].tolist()
boxplot_modelos(ax, df_eval, f"Comparação agregada — {metrica}", metrica, ordem_modelos, CORES)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7. Box plots por grupo de papel
# ============================================================

grupos = sorted(df_eval["grupo_papel"].dropna().unique().tolist())

for grupo in grupos:
    df_g = df_eval[df_eval["grupo_papel"] == grupo]
    ordem = df_g.groupby("modelo")[metrica].median().sort_values().index.tolist()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    boxplot_modelos(
        ax,
        df_g,
        f"Grupo {grupo} — {metrica}",
        metrica,
        ordem,
        CORES,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 8. Grade compacta: todos os grupos + agregado
# ============================================================

def plot_grade_grupos(df_eval, metrica, cores, max_cols=2):
    grupos = ["AGREGADO"] + sorted(df_eval["grupo_papel"].dropna().unique().tolist())
    n = len(grupos)
    ncols = min(max_cols, n)
    nrows = int(np.ceil(n / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.5*nrows), squeeze=False)
    axes = axes.ravel()
    
    for ax, grupo in zip(axes, grupos):
        if grupo == "AGREGADO":
            df_g = df_eval
            titulo = f"Agregado — {metrica}"
        else:
            df_g = df_eval[df_eval["grupo_papel"] == grupo]
            titulo = f"{grupo} — {metrica}"
        
        ordem = df_g.groupby("modelo")[metrica].median().sort_values().index.tolist()
        boxplot_modelos(ax, df_g, titulo, metrica, ordem, cores)
    
    for ax in axes[len(grupos):]:
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

plot_grade_grupos(df_eval, metrica, CORES, max_cols=2)

In [ ]:
# ============================================================
# 9. Exportação opcional das tabelas de resumo
# ============================================================

PASTA_SAIDA = Path("outputs/evaluation_05_boxplots")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

resumo_agregado.to_csv(PASTA_SAIDA / "resumo_agregado.csv", index=False)
resumo_grupo.to_csv(PASTA_SAIDA / "resumo_por_grupo.csv", index=False)

print(f"Arquivos salvos em: {PASTA_SAIDA.resolve()}")